In [1]:
from google.colab import files
uploaded = files.upload()

Saving Telco_customer_churn.xlsx to Telco_customer_churn.xlsx


In [2]:
import pandas as pd

df = pd.read_excel("Telco_customer_churn.xlsx")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 7043
Columns: 33


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [4]:
df.columns.tolist()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [5]:
df['Total Charges'] = pd.to_numeric(
    df['Total Charges'],
    errors='coerce'
)

df['Total Charges'].dtype

dtype('float64')

In [6]:
df.isnull().sum().sort_values(ascending=False)

,0
Churn Reason,5174
Total Charges,11
CustomerID,0
Count,0
Country,0
Zip Code,0
Lat Long,0
State,0
City,0
Gender,0


In [7]:
df[df['Total Charges'].isnull()][
    ['CustomerID','Tenure Months','Monthly Charges','Total Charges']
]

,CustomerID,Tenure Months,Monthly Charges,Total Charges
2234,4472-LVYGI,0,52.55,NaN
2438,3115-CZMZD,0,20.25,NaN
2568,5709-LVOEQ,0,80.85,NaN
2667,4367-NUYAO,0,25.75,NaN
2856,1371-DWPAZ,0,56.05,NaN
4331,7644-OMVMY,0,19.85,NaN
4687,3213-VVOLG,0,25.35,NaN
5104,2520-SGTTA,0,20.00,NaN
5719,2923-ARZLG,0,19.70,NaN
6772,4075-WKNIU,0,73.35,NaN


In [8]:
df['Total Charges'] = pd.to_numeric(
    df['Total Charges'],
    errors='coerce'
)

df['Total Charges'] = df['Total Charges'].fillna(0)

In [9]:
df.isnull().sum().sort_values(ascending=False).head()

,0
Churn Reason,5174
CustomerID,0
Count,0
State,0
Country,0


In [10]:
import sqlite3

conn = sqlite3.connect("customer_churn.db")

df.to_sql(
    "telco_customers",
    conn,
    if_exists="replace",
    index=False
)

print("Database Created Successfully")

Database Created Successfully


In [11]:
query = """
SELECT
    Contract,
    COUNT(*) AS Customers
FROM telco_customers
GROUP BY Contract
ORDER BY Customers DESC
"""

pd.read_sql(query, conn)

,Contract,Customers
0,Month-to-month,3875
1,Two year,1695
2,One year,1473


In [12]:
query = """
SELECT
    Contract,
    COUNT(*) AS Churned_Customers
FROM telco_customers
WHERE `Churn Label` = 'Yes'
GROUP BY Contract
ORDER BY Churned_Customers DESC
"""

pd.read_sql(query, conn)

,Contract,Churned_Customers
0,Month-to-month,1655
1,One year,166
2,Two year,48


In [13]:
query = """
SELECT
    Contract,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END) AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Percent
FROM telco_customers
GROUP BY Contract
ORDER BY Churn_Rate_Percent DESC
"""

pd.read_sql(query, conn)

,Contract,Total_Customers,Churned_Customers,Churn_Rate_Percent
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


Contract type emerged as the strongest business predictor of customer churn.

Month-to-Month customers exhibited a churn rate of 42.71%, compared to 11.27% for One-Year contracts and only 2.83% for Two-Year contracts.

Month-to-Month customers accounted for 88.5% of all churned customers, highlighting contract commitment as a critical retention lever.

In [14]:
query = """
SELECT
    Contract,
    ROUND(SUM(`Total Charges`),2) AS Revenue
FROM telco_customers
GROUP BY Contract
ORDER BY Revenue DESC
"""

pd.read_sql(query, conn)

,Contract,Revenue
0,Two year,6283253.7
1,Month-to-month,5305861.5
2,One year,4467053.5


In [15]:
query = """
SELECT
    Contract,
    ROUND(SUM(`Total Charges`),2) AS Revenue_At_Risk
FROM telco_customers
WHERE `Churn Label`='Yes'
GROUP BY Contract
ORDER BY Revenue_At_Risk DESC
"""

pd.read_sql(query, conn)

,Contract,Revenue_At_Risk
0,Month-to-month,1927182.25
1,One year,674991.20
2,Two year,260753.45


Revenue exposure analysis revealed that Month-to-Month customers account for approximately 67% of total revenue at risk.

Although Two-Year customers generated the highest overall revenue, their churn rate remained extremely low, resulting in significantly lower revenue leakage.

Retention initiatives focused on Month-to-Month customers are likely to generate the highest financial impact.

In [16]:
query = """
SELECT
    `Internet Service`,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END) AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM telco_customers
GROUP BY `Internet Service`
ORDER BY Churn_Rate DESC
"""

pd.read_sql(query, conn)

,Internet Service,Total_Customers,Churned_Customers,Churn_Rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [17]:
query = """
SELECT
    `Tech Support`,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END) AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM telco_customers
GROUP BY `Tech Support`
ORDER BY Churn_Rate DESC
"""

pd.read_sql(query, conn)

,Tech Support,Total_Customers,Churned_Customers,Churn_Rate
0,No,3473,1446,41.64
1,Yes,2044,310,15.17
2,No internet service,1526,113,7.40


In [18]:
query = """
SELECT
    `Online Security`,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END) AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN `Churn Label`='Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM telco_customers
GROUP BY `Online Security`
ORDER BY Churn_Rate DESC
"""

pd.read_sql(query, conn)

,Online Security,Total_Customers,Churned_Customers,Churn_Rate
0,No,3498,1461,41.77
1,Yes,2019,295,14.61
2,No internet service,1526,113,7.40


In [19]:
query = """
SELECT
    CASE
        WHEN `Churn Score` >= 80 THEN 'High Risk'
        WHEN `Churn Score` >= 50 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Category,

    COUNT(*) AS Customers,

    ROUND(AVG(`Monthly Charges`),2) AS Avg_Monthly_Charge,

    ROUND(SUM(`Total Charges`),2) AS Revenue
FROM telco_customers
GROUP BY Risk_Category
ORDER BY Customers DESC
"""

pd.read_sql(query, conn)

,Risk_Category,Customers,Avg_Monthly_Charge,Revenue
0,Medium Risk,3309,64.42,7606217.35
1,Low Risk,2533,60.95,6416210.15
2,High Risk,1201,73.73,2033741.20
